<a href="https://colab.research.google.com/github/18PrajeetR/Mact_Prediction_ML_Pipeline/blob/main/MACT_ML_Project_March_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
MACT (Motor Accidents and Claims Tribunal) - ML Pipeline
=========================================================
Model   : RidgeCV (auto-tuned alpha via cross validation)
Target  : Total_Compensation_Awarded  (log-transformed, Death cases only)
Metrics : RMSE (₹), R², MAPE% — all in rupee scale

Saves   : mact_ridgecv_model.pkl  — trained RidgeCV model
          mact_scaler_death.pkl   — fitted StandardScaler
          mact_feature_names.pkl  — ordered feature list for Streamlit alignment
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")




# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv('/content/MACT_Synthetic_Dataset_India - MACT_Synthetic_Dataset_India.csv.csv')
    print(f"[DATA] Loaded {len(df):,} rows × {df.shape[1]} columns")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 2. PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full preprocessing pipeline returning one cleaned dataset:
    Death cases only. Injury cases are excluded from this model.
    """
    # ── 2a. Drop redundant / zero-signal columns ──────────────────────────────
    # Multiplier and Future_Prospects_Pct are deterministically derived from Age
    # Dependent_Parents has r ≈ 0.001 with target
    # S_No is just a row index
    drop_cols = ["S_No", "Multiplier", "Future_Prospects_Pct", "Dependent_Parents"]
    df = df.drop(columns=drop_cols)
    print(f"[PREP] Dropped {drop_cols}")

    # ── 2b. Keep Death cases only ─────────────────────────────────────────────
    df_death = df[df["Case_Type"] == "Death"].copy().reset_index(drop=True)
    df_death = df_death.drop(columns=["Case_Type"])
    print(f"[PREP] Death cases retained: {len(df_death):,}")
    print(f"[PREP] Injury cases removed: model is Death-only by design")

    # ── 2c. Log-transform target ──────────────────────────────────────────────
    # Compensation is right-skewed (skew=1.22); log brings it to near-normal
    df_death["log_compensation"] = np.log(df_death["Total_Compensation_Awarded"])
    df_death.drop(columns=["Total_Compensation_Awarded"], inplace=True)

    # ── 2d. One-hot encode categorical features ───────────────────────────────
    cat_cols = ["Gender", "Marital_status", "Employment_Type",
                "Vehicle_Hit", "Insurance_Valid", "State"]

    df_death = pd.get_dummies(df_death, columns=cat_cols, drop_first=True)
    print(f"[PREP] Feature columns after encoding: {df_death.shape[1] - 1}")

    return df_death


# ══════════════════════════════════════════════════════════════════════════════
# 3. TRAIN / TEST SPLIT + SCALING
# ══════════════════════════════════════════════════════════════════════════════
def split_and_scale(df: pd.DataFrame, target_col: str = "log_compensation",
                    test_size: float = 0.2, random_state: int = 42):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Scale only numeric columns (tree models don't strictly need it, but
    # Ridge does, and it never hurts XGBoost/RF)
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

    return X_train, X_test, y_train, y_test, scaler


# ══════════════════════════════════════════════════════════════════════════════
# 4. METRICS HELPER
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, X_test, y_test, model_name: str):
    """
    Evaluates model performance in rupee scale only.
    Single standard form: RMSE (₹), R², MAPE%.
    """
    y_pred_log = model.predict(X_test)

    # ── Convert back to rupee scale ───────────────────────────────────────────
    y_test_rupee = np.exp(y_test)
    y_pred_rupee = np.exp(y_pred_log)

    # ── Compute all three metrics in rupee scale ──────────────────────────────
    rmse = np.sqrt(mean_squared_error(y_test_rupee, y_pred_rupee))
    r2   = r2_score(y_test_rupee, y_pred_rupee)
    mape = np.mean(np.abs((y_test_rupee - y_pred_rupee) / y_test_rupee)) * 100

    print(f"\n  {'─'*50}")
    print(f"  Model : {model_name}")
    print(f"  {'─'*50}")
    print(f"  RMSE  : ₹{rmse:,.0f}")
    print(f"  R²    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")
    print(f"  {'─'*50}")

    return {
        "model":   model_name,
        "rmse":    round(rmse, 0),
        "r2":      round(r2, 4),
        "mape":    round(mape, 2),
    }


# ══════════════════════════════════════════════════════════════════════════════
# 5. MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_model() -> RidgeCV:
    # RidgeCV tests each alpha value using 5-fold cross validation internally
    # and automatically keeps the one that produced the lowest CV error.
    # No guessing. No manual tuning needed.
    return RidgeCV(
        alphas=[0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0, 1000.0],
        cv=5
    )


# ══════════════════════════════════════════════════════════════════════════════
# 6. CROSS-VALIDATION HELPER
# ══════════════════════════════════════════════════════════════════════════════
def cross_validate_model(model, X_train, y_train, model_name: str,
                          case_type: str, cv: int = 5):
    """5-fold CV on training data to catch overfitting early."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train,
                             scoring="r2", cv=kf, n_jobs=-1)
    print(f"  [CV] {model_name} [{case_type}] "
          f"R² per fold: {np.round(scores, 3)} | "
          f"Mean: {scores.mean():.3f} ± {scores.std():.3f}")





# ══════════════════════════════════════════════════════════════════════════════
# 8. MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
def run_pipeline(filepath: str):
    print("\n" + "═" * 60)
    print("  MACT ML PIPELINE — RidgeCV (Death Cases)")
    print("═" * 60)

    # ── Load & preprocess ─────────────────────────────────────────────────────
    df       = load_data(filepath)
    df_death = preprocess(df)
    model    = build_model()

    # ── Split and scale ───────────────────────────────────────────────────────
    X_train, X_test, y_train, y_test, scaler = split_and_scale(df_death)
    feature_names = X_train.columns.tolist()
    print(f"\n[SPLIT] Train: {len(X_train):,} | Test: {len(X_test):,}")

    # ── Cross-validate on training data ──────────────────────────────────────
    print(f"\n[CV] Running 5-fold CV for RidgeCV...")
    cross_validate_model(model, X_train, y_train, "RidgeCV", "Death")

    # ── Fit on full training set ──────────────────────────────────────────────
    model.fit(X_train, y_train)
    print(f"\n[TRAIN] Best alpha selected by RidgeCV: {model.alpha_:.4f}")

    # ── Evaluate on held-out test set ─────────────────────────────────────────
    result = evaluate(model, X_test, y_test, "RidgeCV (auto-tuned)")

    # ── Save all three artefacts needed for Streamlit ─────────────────────────
    joblib.dump(model,         "mact_ridgecv_model.pkl")
    joblib.dump(scaler,        "mact_scaler_death.pkl")
    joblib.dump(feature_names, "mact_feature_names.pkl")
    print(f"\n[SAVED] mact_ridgecv_model.pkl  ← trained RidgeCV model")
    print(f"[SAVED] mact_scaler_death.pkl   ← fitted StandardScaler")
    print(f"[SAVED] mact_feature_names.pkl  ← {len(feature_names)} feature names for Streamlit")

    # ── Final summary ─────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print("  FINAL RESULT")
    print(f"{'═'*60}")
    print(f"  Model : RidgeCV (auto-tuned)")
    print(f"  Alpha : {model.alpha_:.4f}  (auto-selected from 8 candidates)")
    print(f"  R²    : {result['r2']:.4f}")
    print(f"  RMSE  : ₹{result['rmse']:,.0f}")
    print(f"  MAPE  : {result['mape']:.2f}%")
    print(f"{'═'*60}")
    print("  Pipeline complete. 3 pkl files ready for Streamlit.")
    print(f"{'═'*60}\n")

    return result


# ══════════════════════════════════════════════════════════════════════════════
# 9. PREDICTION HELPER (for inference on new cases)
# ══════════════════════════════════════════════════════════════════════════════
def predict_compensation(model, scaler, new_data: pd.DataFrame,
                          feature_names: list) -> np.ndarray:
    """
    Pass in a DataFrame of new cases (already one-hot encoded),
    returns predicted compensation in rupees.

    Example usage:
        new = pd.DataFrame([{
            "Age": 35, "Monthly_income": 45000,
            "Number_of_Dependents": 2, "Negligence_Percentage": 10,
            "Deduction_Fraction": 0.33, "Year": 2023,
            "Gender_Male": 1, "Insurance_Valid_Yes": 1,
            ... (all one-hot columns)
        }])
        rupee_pred = predict_compensation(best_model, scaler, new, feature_names)
    """
    new_aligned = new_data.reindex(columns=feature_names, fill_value=0)
    numeric_cols = new_aligned.select_dtypes(include=[np.number]).columns
    new_aligned[numeric_cols] = scaler.transform(new_aligned[numeric_cols])
    log_pred  = model.predict(new_aligned)
    rupee_pred = np.exp(log_pred)
    return rupee_pred


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import sys

    # Default path — update if your file is elsewhere
    DATA_PATH = (
        sys.argv[1] if len(sys.argv) > 1
        else "MACT_Synthetic_Dataset_India_-_MACT_Synthetic_Dataset_India_csv.csv"
    )

    summary = run_pipeline(DATA_PATH)


════════════════════════════════════════════════════════════
  MACT ML PIPELINE — RidgeCV (Death Cases)
════════════════════════════════════════════════════════════
[DATA] Loaded 12,313 rows × 18 columns
[PREP] Dropped ['S_No', 'Multiplier', 'Future_Prospects_Pct', 'Dependent_Parents']
[PREP] Death cases retained: 8,050
[PREP] Injury cases removed: model is Death-only by design
[PREP] Feature columns after encoding: 36

[SPLIT] Train: 6,440 | Test: 1,610

[CV] Running 5-fold CV for RidgeCV...
  [CV] RidgeCV [Death] R² per fold: [0.926 0.925 0.927 0.931 0.927] | Mean: 0.927 ± 0.002

[TRAIN] Best alpha selected by RidgeCV: 10.0000

  ──────────────────────────────────────────────────
  Model : RidgeCV (auto-tuned)
  ──────────────────────────────────────────────────
  RMSE  : ₹1,227,273
  R²    : 0.8943
  MAPE  : 14.03%
  ──────────────────────────────────────────────────

[SAVED] mact_ridgecv_model.pkl  ← trained RidgeCV model
[SAVED] mact_scaler_death.pkl   ← fitted StandardScaler
[SA